# 🛰️ xView2 Building Damage Assessment Training
This notebook trains a YOLO segmentation model to detect building damage from satellite imagery.

In [ ]:
# 1. Setup Environment
!pip install ultralytics
!pip install -r ai/training/colab/requirements.txt

import os
from ultralytics import YOLO
import matplotlib.pyplot as plt
from google.colab import drive

print("✅ Environment setup complete.")

In [ ]:
# 2. Mount Google Drive & Load Data
drive.mount('/content/drive')

# CHANGE THESE PATHS to match your Drive folders
ZIP_PATH = "/content/drive/MyDrive/xview2_processed/multi_hazards.zip"
YAML_PATH = "/content/drive/MyDrive/xview2_processed/dataset.yaml"

# Unzip to local Colab disk for significantly faster training I/O
!unzip -q {ZIP_PATH} -d /content/dataset

print("✅ Dataset extracted to /content/dataset")

In [ ]:
# 3. Initialize Model
# We use YOLO11n-seg (Nano) for high efficiency and feasibility
model = YOLO('yolo11n-seg.pt') 

print("✅ Model initialized.")

In [ ]:
# 4. Train the Model
# Hyperparameters are passed here. We can also use the yaml file via cfg="."
results = model.train(
    data=YAML_PATH,
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    name='disaster_damage_model'
)

print("✅ Training complete.")

In [ ]:
# 5. Evaluation & Visualization
metrics = model.val()
print(f"mAP50-95: {metrics.seg.map}")
print(f"mAP50: {metrics.seg.map50}")

# Plot results
plt.imshow(metrics.confusion_matrix)
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# 6. Export Weights
best_weights_path = "runs/segment/disaster_damage_model/weights/best.pt"
save_path = "/content/drive/MyDrive/xview2_processed/best.pt"

!cp {best_weights_path} {save_path}

print(f"✅ Weights saved to {save_path}")